# Texas State Employee Salary Prediction
**Project ID:** PRCP-1024-TexasSalaryPrediction

## Project objectives
1. Prepare a complete data analysis report.
2. Build a predictive model to estimate annual payroll salary.
3. Identify salary outliers.
4. Explore wage differences between manager-labelled and other roles across agencies.
5. Examine salary, compensation and head-count patterns over time where the supplied data supports it.
6. Compare multiple models, recommend a production model, and document challenges.

**Dataset:** `salary.csv` (provided with this project).

> **Important data limitation:** The supplied file is a salary-record snapshot and contains `EMPLOY DATE`, but no salary-record year, historical payroll period, or annual time-series field. Therefore, this notebook can analyze employee start-date cohorts, but it cannot prove how salaries or head-count changed year by year. Manager status is also not a dedicated field; manager roles below are identified using title keywords and should be treated as an approximation.

## 1. Import required libraries

Run this notebook from the folder containing `salary.csv`. If needed, install missing packages with `!pip install pandas numpy matplotlib seaborn scikit-learn openpyxl`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Load the supplied dataset

In [ ]:
df = pd.read_csv('salary.csv', low_memory=False)

print('Dataset shape:', df.shape)
display(df.head())

## 3. Understand the dataset

In [ ]:
df.info()
display(df.describe(include='all').T)

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print('Duplicate rows:', df.duplicated().sum())

missing = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing %': (df.isna().mean() * 100).round(2)
}).sort_values('Missing %', ascending=False)

display(missing)

## 4. Data cleaning and preparation

Text fields are trimmed to remove padding. Salary and working-hour fields are converted to numeric values. The employee date is parsed as a date. Original records are retained in `df`; the cleaned analysis copy is `data`.

In [ ]:
data = df.copy()

text_columns = data.select_dtypes(include=['object']).columns
for column in text_columns:
    data[column] = data[column].astype('string').str.strip()
    data[column] = data[column].replace('', pd.NA)

numeric_columns = ['AGENCY', 'HRLY RATE', 'HRS PER WK', 'MONTHLY',
                   'ANNUAL', 'STATE NUMBER', 'summed_annual_salary']
for column in numeric_columns:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors='coerce')

data['EMPLOY DATE'] = pd.to_datetime(data['EMPLOY DATE'], errors='coerce')
data['EMPLOY YEAR'] = data['EMPLOY DATE'].dt.year

print('Cleaned shape:', data.shape)
display(data.head())

## 5. Data quality checks

In [ ]:
print('Missing values after cleaning:')
display(data.isna().sum().sort_values(ascending=False).to_frame('Missing values'))

print('Annual salary summary:')
display(data['ANNUAL'].describe())

print('Non-positive annual salary rows:', (data['ANNUAL'] <= 0).sum())
print('Invalid / missing employment dates:', data['EMPLOY DATE'].isna().sum())

## 6. Exploratory Data Analysis (EDA)

### 6.1 Annual salary distribution

In [ ]:
salary_data = data[data['ANNUAL'].notna() & (data['ANNUAL'] > 0)].copy()

plt.figure(figsize=(10, 5))
sns.histplot(salary_data['ANNUAL'], bins=60, kde=True)
plt.title('Distribution of Annual Salaries')
plt.xlabel('Annual Salary')
plt.ylabel('Number of Records')
plt.show()

plt.figure(figsize=(10, 3))
sns.boxplot(x=salary_data['ANNUAL'])
plt.title('Annual Salary Boxplot')
plt.show()

### 6.2 Salary by agency

In [ ]:
agency_salary = (salary_data.groupby('AGENCY NAME', dropna=False)['ANNUAL']
                .agg(['count', 'mean', 'median'])
                .sort_values('median', ascending=False))
display(agency_salary.head(15))

top_agencies = salary_data['AGENCY NAME'].value_counts().head(12).index
plt.figure(figsize=(13, 6))
sns.boxplot(data=salary_data[salary_data['AGENCY NAME'].isin(top_agencies)],
            x='AGENCY NAME', y='ANNUAL')
plt.xticks(rotation=75, ha='right')
plt.title('Annual Salary by Agencies with the Most Records')
plt.tight_layout()
plt.show()

### 6.3 Salary by job title

In [ ]:
title_salary = (salary_data.groupby('CLASS TITLE', dropna=False)['ANNUAL']
               .agg(['count', 'mean', 'median'])
               .query('count >= 20')
               .sort_values('median', ascending=False))
display(title_salary.head(20))

display(title_salary.sort_values('median').head(20))

### 6.4 Salary by gender and ethnicity

In [ ]:
display(salary_data.groupby('GENDER', dropna=False)['ANNUAL']
        .agg(['count', 'mean', 'median']).sort_values('median', ascending=False))

display(salary_data.groupby('ETHNICITY', dropna=False)['ANNUAL']
        .agg(['count', 'mean', 'median']).sort_values('median', ascending=False))

These are descriptive, unadjusted comparisons. They do not establish that gender or ethnicity causes salary differences. Job title, agency, experience, hours and employment status may differ between groups.

## 7. Salary outlier analysis

In [ ]:
q1 = salary_data['ANNUAL'].quantile(0.25)
q3 = salary_data['ANNUAL'].quantile(0.75)
iqr = q3 - q1
lower_limit = max(0, q1 - 1.5 * iqr)
upper_limit = q3 + 1.5 * iqr

outliers = salary_data[
    (salary_data['ANNUAL'] < lower_limit) |
    (salary_data['ANNUAL'] > upper_limit)
].copy()

print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower fence:', lower_limit)
print('Upper fence:', upper_limit)
print('Outlier count:', len(outliers))
display(outliers[['AGENCY NAME', 'CLASS TITLE', 'STATUS',
                  'HRLY RATE', 'HRS PER WK', 'ANNUAL']]
        .sort_values('ANNUAL', ascending=False).head(30))

The IQR method flags unusually low or high records; it does not prove an error or misconduct. Part-time schedules, specialized positions, multiple jobs and unusual employment arrangements can produce legitimate extreme values.

## 8. Wage disparities between manager-labelled and other roles

In [ ]:
# Manager is not supplied as a separate field. Use transparent title-keyword approximation.
manager_pattern = r'\b(manager|director|chief|superintendent|administrator|supervisor)\b'
salary_data['ROLE_GROUP'] = np.where(
    salary_data['CLASS TITLE'].str.contains(manager_pattern, case=False, na=False),
    'Manager-labelled title',
    'Other title'
)

role_comparison = (salary_data.groupby(['AGENCY NAME', 'ROLE_GROUP'])['ANNUAL']
                   .agg(['count', 'mean', 'median']).reset_index())
display(role_comparison.head(20))

role_pivot = role_comparison.pivot(index='AGENCY NAME',
                                   columns='ROLE_GROUP', values='median')
role_pivot['Median Pay Gap'] = (
    role_pivot.get('Manager-labelled title') - role_pivot.get('Other title')
)
display(role_pivot.sort_values('Median Pay Gap', ascending=False).head(20))

This is a title-keyword proxy, not an official management classification. Agencies with small group sizes can show unstable differences; inspect counts before interpreting a gap.

## 9. Employment-date cohort and head-count analysis

In [ ]:
cohort_data = data[data['EMPLOY YEAR'].notna()].copy()

yearly_headcount = cohort_data.groupby('EMPLOY YEAR')['STATE NUMBER'].nunique()
yearly_starts = cohort_data.groupby('EMPLOY YEAR').size()

fig, ax = plt.subplots(figsize=(12, 5))
yearly_starts.plot(kind='bar', ax=ax)
ax.set_title('Number of Records by Employee Start Year')
ax.set_xlabel('Employment Start Year')
ax.set_ylabel('Record Count')
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    'Employee IDs (unique) with this start year': yearly_headcount,
    'Salary records with this start year': yearly_starts
}).tail(25))

**Interpretation limit:** These charts group currently listed records by their employment start year. They are not historical annual head-count or payroll totals. Employees who left before the snapshot may be absent, and there is no year-specific salary observation in the supplied file.

## 10. Prepare data for salary prediction

In [ ]:
# Predict annual salary at the job-record level.
# Exclude direct identifiers and fields that could leak salary or duplicate it.
target = 'ANNUAL'
excluded_columns = [
    target, 'LAST NAME', 'FIRST NAME', 'MI', 'STATE NUMBER',
    'summed_annual_salary', 'combined_multiple_jobs',
    'duplicated', 'multiple_full_time_jobs', 'hide_from_search',
    'EMPLOY DATE'
]
features = data.drop(columns=[c for c in excluded_columns if c in data.columns]).copy()

# EMPLOY YEAR is retained as a coarse tenure/start-cohort feature.
features = features.drop(columns=['ANNUAL'], errors='ignore')

model_data = features.copy()
model_data[target] = data[target]

# Keep rows with a valid, positive salary target.
model_data = model_data[model_data[target].notna() & (model_data[target] > 0)].copy()

X = model_data.drop(columns=[target])
y = model_data[target]

# Split by employee identifier so multiple job records for one employee
# do not appear in both train and test sets.
groups = data.loc[model_data.index, 'STATE NUMBER'].fillna(-1)

print('Model rows:', len(model_data))
print('Feature columns:', X.columns.tolist())
print('Target summary:')
display(y.describe())

## 11. Train-test split by employee

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_index, test_index = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()
y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 12. Preprocessing pipelines

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=30))
])

linear_preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_preprocessor, numeric_features),
    ('categorical', categorical_preprocessor, categorical_features)
])

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 13. Train and compare multiple models

In [ ]:
models = {
    'Dummy Baseline': Pipeline(steps=[\n        ('preprocess', linear_preprocessor),\n        ('model', DummyRegressor(strategy='median'))\n    ]),
    'Ridge Regression': Pipeline(steps=[
        ('preprocess', linear_preprocessor),
        ('model', Ridge(alpha=10))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocess', linear_preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=120, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        ))
    ])
}

# HistGradientBoosting uses ordinal-encoded categorical values.
tree_preprocessor = ColumnTransformer(transformers=[
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('categorical', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value',
                                  unknown_value=-1))
    ]), categorical_features)
])

models['Hist Gradient Boosting'] = Pipeline(steps=[
    ('preprocess', tree_preprocessor),
    ('model', HistGradientBoostingRegressor(
        max_iter=120, learning_rate=0.08,
        max_leaf_nodes=31, random_state=42
    ))
])

results = []
trained_models = {}

for name, model in models.items():
    print('Training:', name)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })
    trained_models[name] = model

comparison = pd.DataFrame(results).sort_values('MAE')
display(comparison)

## 14. Select the best model

In [ ]:
best_model_name = comparison.iloc[0]['Model']
best_model = trained_models[best_model_name]

print('Best model based on lowest test MAE:', best_model_name)
print('Test MAE:', comparison.iloc[0]['MAE'])
print('Test RMSE:', comparison.iloc[0]['RMSE'])
print('Test R2:', comparison.iloc[0]['R2 Score'])

**Production recommendation:** Use the model with the lowest hold-out MAE as the initial candidate, not an automatic production deployment. Validate it with payroll subject-matter experts, check errors across agencies and job groups, confirm the intended prediction use, monitor drift, and review fairness and privacy. The final winner is determined only after the notebook runs on the supplied data.

## 15. Actual vs predicted salary

In [ ]:
test_predictions = best_model.predict(X_test)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, test_predictions, alpha=0.25)
min_value = min(y_test.min(), test_predictions.min())
max_value = max(y_test.max(), test_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], 'r--')
plt.xlabel('Actual Annual Salary')
plt.ylabel('Predicted Annual Salary')
plt.title('Actual vs Predicted Annual Salary')
plt.tight_layout()
plt.show()

residuals = y_test - test_predictions
plt.figure(figsize=(10, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Prediction Residual Distribution')
plt.xlabel('Actual - Predicted')
plt.show()

## 16. Error analysis by agency

In [ ]:
error_report = X_test.copy()
error_report['Actual Salary'] = y_test
error_report['Predicted Salary'] = test_predictions
error_report['Absolute Error'] = np.abs(
    error_report['Actual Salary'] - error_report['Predicted Salary']
)

agency_errors = (error_report.groupby('AGENCY NAME')['Absolute Error']
                 .agg(['count', 'mean', 'median'])
                 .sort_values('mean', ascending=False))
display(agency_errors.head(20))

## 17. Example: predict salary for a new record

In [ ]:
# Example values are placeholders. Replace them with valid values from your dataset.
# Do not include names or employee identifiers.
example_record = X_train.iloc[[0]].copy()
example_prediction = best_model.predict(example_record)[0]

print('Example input:')
display(example_record)
print(f'Predicted annual salary: {example_prediction:,.2f}')

## 18. Challenges faced and techniques used

In [ ]:
challenges = pd.DataFrame([
    ['Mixed data types', 'Some columns contain mixed or padded text values.',
     'Read with low_memory=False, trim text and convert numeric columns explicitly.'],
    ['Missing values', 'Some attributes contain null values.',
     'Measure missingness and use median / most-frequent imputers inside pipelines.'],
    ['Multiple job records', 'One employee can appear in more than one job record.',
     'Exclude employee ID from model features and split train/test by employee group.'],
    ['High-cardinality categories', 'Agency and job-title categories can create many encoded columns.',
     'Use OneHotEncoder with min_frequency and unknown-category handling.'],
    ['Salary outliers', 'Some salary values are far from the central distribution.',
     'Use IQR to flag and inspect records; do not automatically delete valid high salaries.'],
    ['Manager classification', 'No explicit manager indicator is supplied.',
     'Use transparent title-keyword grouping and label the result as approximate.'],
    ['No historical payroll period', 'The dataset does not contain annual salary snapshots over time.',
     'Analyze employment start-year cohorts only; do not claim historical salary trends.'],
    ['Privacy and leakage', 'Names and employee IDs identify individuals; derived salary fields may leak target information.',
     'Remove identifiers and salary-derived columns from predictors.']
], columns=['Challenge', 'Reason', 'Technique / justification'])

display(challenges)

## 19. Final project conclusion

In [ ]:
print('FINAL MODEL COMPARISON')
display(comparison)

print('Selected model:', best_model_name)
print('The selected model is the lowest-MAE model on the employee-grouped hold-out test set.')
print('Review the EDA, outlier table, role comparison, cohort analysis and error report before final submission.')

### Conclusion template for the project report

This project analyzed the Texas state employee salary dataset and developed regression models to estimate annual salary. The analysis covered salary distributions, agency and job-title comparisons, demographic summaries, IQR-based salary outliers, and an approximate manager-title versus other-title comparison. Multiple regression models were evaluated using an employee-grouped train/test split and compared with MAE, RMSE and R². The model with the lowest test MAE was selected as the leading candidate. The analysis also identified important limitations: the dataset does not contain historical salary snapshots, manager status is not explicitly defined, and employee records may represent multiple jobs. Before production use, the selected model should undergo agency-level error review, fairness and privacy checks, validation by payroll experts, and ongoing monitoring.

**Submission reminder:** Run every cell from top to bottom, review the generated tables and charts, and save the executed notebook with its outputs.